# deprecated; use main.py instead..

### Imports

In [ ]:
# Import all necessary modules from the local imports module
from imports import *

In [ ]:
# dataset
from make_dataset import all_samples_train, all_samples_val
# scoring model
from models import PathScoringModel, contrastive_loss
# dataloading
from dataloaders import PathPairDataset, collate_fn, pad_and_stack

In [ ]:
# Seed
torch.manual_seed(42)
torch.cuda.manual_seed(42)
np.random.seed(42)

### Load Dataset

In [ ]:
train_dataloader = DataLoader(
    PathPairDataset(all_samples_train),
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)
val_dataloader = DataLoader(
    PathPairDataset(all_samples_val),
    batch_size=8,
    shuffle=True,
    collate_fn=collate_fn
)

Sanity Check

In [ ]:
# # Example forward pass
# model = PathScoringModel()
# for batch in train_dataloader:
#     scores = model.forward(batch['good_feats'], batch['good_len'], batch['bad_feats'], batch['bad_len'])
#     print("Scores shape:", scores[0].shape, scores[1].shape)  # [B]
#     break  # Just to see one batch

### Load Model

In [ ]:
# initialize the model
model = PathScoringModel()

# learning parameters
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
num_epochs = 20

Sanity Check

In [ ]:
# # Example training step
# for batch in train_dataloader:
#     good_path_feats, good_lens = batch['good_feats'], batch['good_len']
#     bad_path_feats, bad_lens = batch['bad_feats'], batch['bad_len']

#     score_good, score_bad = model(good_path_feats, good_lens, bad_path_feats, bad_lens)
#     loss = contrastive_loss(score_good, score_bad)

#     loss.backward()
#     optimizer.step()

#### What happens when we shuffle labels in the training set?
Can we interpolate random data?

In [ ]:
# all_samples_train_shuffled = []
# for phrase in all_samples_train:
#     goods = len(phrase["good_paths"])
#     bads = len(phrase["bad_paths"])
#     alls = goods + bads
#     all_paths = phrase["good_paths"] + phrase["bad_paths"]
#     new_good_indices = np.random.choice(range(alls), size=goods, replace=False)
#     new_good_paths = [all_paths[ii] for ii in range(alls) if ii in new_good_indices]
#     new_bad_paths = [all_paths[ii] for ii in range(alls) if ii not in new_good_indices]
#     all_samples_train_shuffled.append({
#         "good_paths": new_good_paths,
#         "bad_paths": new_bad_paths,
#     })

# all_samples_train = all_samples_train_shuffled

### Train

In [ ]:
# full training loop

train_losses = []
val_losses = []

for epoch in range(num_epochs):
    model.train()
    total_loss = 0.0
    for batch in train_dataloader:
        optimizer.zero_grad()
        
        good_path_feats, good_lens = batch['good_feats'], batch['good_len']
        bad_path_feats, bad_lens = batch['bad_feats'], batch['bad_len']

        score_good, score_bad = model(good_path_feats, good_lens, bad_path_feats, bad_lens)
        loss = contrastive_loss(score_good, score_bad)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()
        
    model.eval()
    val_loss = 0.0
    with torch.no_grad():
        for val_batch in val_dataloader:
            val_good_feats, val_good_lens = val_batch['good_feats'], val_batch['good_len']
            val_bad_feats, val_bad_lens = val_batch['bad_feats'], val_batch['bad_len']
            
            val_score_good, val_score_bad = model(val_good_feats, val_good_lens, val_bad_feats, val_bad_lens)
            val_loss += contrastive_loss(val_score_good, val_score_bad).item()

    print(f"Epoch {epoch+1}/{num_epochs}, Train Loss: {total_loss/len(train_dataloader)}, Val Loss: {val_loss / len(val_dataloader)}")
    train_losses.append(total_loss/len(train_dataloader))
    val_losses.append(val_loss/len(val_dataloader))

In [ ]:
# plot the training and validation losses
plt.plot(train_losses, label='Train Loss')
plt.plot(val_losses, label='Validation Loss')
plt.legend()
plt.ylabel('Contrastive Loss')
plt.xlabel('Epoch')

### Eval

In [ ]:
# get scores for all paths
good_path_scores = []
bad_path_scores = []
for batch in val_dataloader:
    good_path_feats, good_lens = batch["good_feats"], batch["good_len"]
    bad_path_feats, bad_lens = batch["bad_feats"], batch["bad_len"]

    score_good, score_bad = model(good_path_feats, good_lens, bad_path_feats, bad_lens)
    good_path_scores.extend(score_good.detach().numpy())
    bad_path_scores.extend(score_bad.detach().numpy())

In [ ]:
plt.scatter(good_path_scores, np.ones(len(good_path_scores)), label='Good Path Scores', marker='o')
plt.scatter(bad_path_scores, np.zeros(len(bad_path_scores)), label='Bad Path Scores', marker='x')
plt.xlabel('Predicted Path Score')
plt.ylabel('Ground truth Path Score')
plt.ylim(-1, 2)
plt.yticks([0, 1])

In [ ]:
# measure cross entropy loss using good_path_scores, bad_path_scores
# good_path_scores have label 1, bad_path_scores have label 0
good_labels = torch.ones(len(good_path_scores))
bad_labels = torch.zeros(len(bad_path_scores))
all_scores = torch.tensor(good_path_scores + bad_path_scores, dtype=torch.float32)
all_labels = torch.cat([good_labels, bad_labels])
print(f"Classification AUC: {roc_auc_score(all_labels.numpy(), all_scores.numpy())}")


In [ ]:
# plot roc curve
from sklearn.metrics import roc_curve
fpr, tpr, thresholds = roc_curve(all_labels.numpy(), all_scores.numpy())
plt.figure(figsize=(8, 6))
plt.plot(fpr, tpr, label='ROC Curve')
plt.plot([0, 1], [0, 1], 'k--', label='Random Guessing')
plt.xlabel('False Positive Rate')
plt.ylabel('True Positive Rate')
plt.title('ROC Curve')
plt.legend()
plt.show()

### Save/Load Model

In [ ]:
torch.save(model.state_dict(), "save_models/path_scoring_model_v1.pth")

In [ ]:
model_load = PathScoringModel()
model_load.load_state_dict(torch.load("save_models/path_scoring_model_v1.pth"))
model_load.eval()

### Inspect Model

In [ ]:
trial_paths = [
    [(0, 0, 1.0), (-1, 0, 1.33333)],  # S m
    [(0, 0, 1.0), (3, 2, 1.31836)],  # S m
]

In [ ]:
# trial_paths = [
#     encode_hf(coordinate, saptak_mark)
#     for coordinate, saptak_mark in []
# ]

In [ ]:
model.forward_path(torch.tensor(trial_paths, dtype=torch.float32), torch.tensor([len(p) for p in trial_paths]))

Probe the model's GRU layer..

In [ ]:
# pass all samples through the model and get the hidden state
gru_hidden_states_good = []
gru_hidden_states_bad = []

for traj in all_samples_train:
    good_paths = traj["good_paths"]
    bad_paths = traj["bad_paths"]
    good_feats, good_lens = pad_and_stack(good_paths)
    bad_feats, bad_lens = pad_and_stack(bad_paths)
    with torch.no_grad():
        _, h_good = model.path_encoder(model.node_proj(good_feats))
        _, h_bad = model.path_encoder(model.node_proj(bad_feats))
    gru_hidden_states_good.extend(h_good.squeeze(0).numpy())
    gru_hidden_states_bad.extend(h_bad.squeeze(0).numpy())
    
gru_hidden_states_good = np.array(gru_hidden_states_good)
gru_hidden_states_bad = np.array(gru_hidden_states_bad)

In [ ]:
# UMAP import (conditionally available)
if UMAP_AVAILABLE:
    import umap
else:
    print("UMAP not available - visualization will be limited")

In [ ]:
# Fit UMAP on training data
all_hidden_states = np.vstack([gru_hidden_states_good, gru_hidden_states_bad])
umap_model = umap.UMAP(n_components=2, random_state=42)
umap_embeddings = umap_model.fit_transform(all_hidden_states)

In [ ]:
umap_good = umap_embeddings[:len(gru_hidden_states_good)]
umap_bad = umap_embeddings[len(gru_hidden_states_good):]
plt.scatter(umap_good[:, 0], umap_good[:, 1], label='Good Paths', marker='o')
plt.scatter(umap_bad[:, 0], umap_bad[:, 1], label='Bad Paths', marker='x')

In [ ]:
gru_hidden_states_good_val = []
gru_hidden_states_bad_val = []

for traj in all_samples_val:
    good_paths = traj["good_paths"]
    bad_paths = traj["bad_paths"]
    good_feats, good_lens = pad_and_stack(good_paths)
    bad_feats, bad_lens = pad_and_stack(bad_paths)
    with torch.no_grad():
        _, h_good = model.path_encoder(model.node_proj(good_feats))
        _, h_bad = model.path_encoder(model.node_proj(bad_feats))
    gru_hidden_states_good_val.extend(h_good.squeeze(0).numpy())
    gru_hidden_states_bad_val.extend(h_bad.squeeze(0).numpy())
    
gru_hidden_states_good_val = np.array(gru_hidden_states_good_val)
gru_hidden_states_bad_val = np.array(gru_hidden_states_bad_val)

In [ ]:
umap_good_val = umap_model.transform(gru_hidden_states_good_val)
umap_bad_val = umap_model.transform(gru_hidden_states_bad_val)
plt.scatter(umap_good_val[:, 0], umap_good_val[:, 1], label='Good Paths', marker='o')
plt.scatter(umap_bad_val[:, 0], umap_bad_val[:, 1], label='Bad Paths', marker='x')